# Arm-3 backbone comparison — 9 architectures × 4 examples

One harness, one seed, one budget. Data loading, channel construction, target definition,
split, training schedule and the free-running overlap-add rollout are **identical for every
run** — only the sequence backbone changes, so differences are attributable to the
architecture rather than the pipeline.

| | |
|---|---|
| examples | SDOF · 3-storey frame · buried tunnel · wind turbine |
| backbones | GRU, LSTM, Transformer, Mamba-style, Encoder–decoder, TCN, 1D CNN, ResNet, MLP |
| seed | **42** (single seed — see the caveat at the bottom) |
| epochs | **10** |
| training data | **100 %** of the 70 % train split |
| split | 70 / 15 / 15, model selection on val, **metrics on the held-out TEST split** |
| target | Arm 3 residual, `u_nl − u_red`; reconstruction `u_red + σ·Δ̂` |
| runs | 4 × 9 = **36** |

**Requires a GPU runtime.** Runtime → Change runtime type → T4 (or better).

The run loop is resumable: each run writes its own JSON and finished runs are skipped, so
if Colab disconnects just re-run the loop cell.


## 1 · GPU check

Stop here if this reports CPU — on CPU the whole sweep takes ~1.5 h instead of minutes.


In [ ]:
import torch, platform
print("torch      :", torch.__version__)
print("python     :", platform.python_version())
print("cuda avail :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu        :", torch.cuda.get_device_name(0))
    print("memory     : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
else:
    print("\n!! No GPU. Runtime -> Change runtime type -> T4 GPU, then re-run.")


## 2 · Point at the data

Upload the `colab_arch/` folder to your Drive (keep the `data/` subfolder intact), then set
`BASE` below to wherever you put it. Results are written next to it in `results/`.


In [ ]:
import os

from google.colab import drive
drive.mount("/content/drive")

# ── edit this one line if you put the folder somewhere else ──
BASE = "/content/drive/MyDrive/colab_arch"

DATA = os.path.join(BASE, "data")
OUTD = os.path.join(BASE, "results")
os.makedirs(OUTD, exist_ok=True)

DB_PATHS = {"sdof":    os.path.join(DATA, "sdof_eta60_n1500.npz"),
            "frame3":  os.path.join(DATA, "frame3_v3_arm_database.npz"),
            "tunnel":  os.path.join(DATA, "tunnel_moving_train_database.npz"),
            "turbine": os.path.join(DATA, "wt_v20_full.npz")}

missing = [k for k, p in DB_PATHS.items() if not os.path.exists(p)]
assert not missing, f"missing databases {missing} under {DATA}\n  found: {os.listdir(DATA) if os.path.isdir(DATA) else 'no data/ dir'}"
for k, p in DB_PATHS.items():
    print(f"  {k:8s} {os.path.getsize(p)/1e6:7.1f} MB  {p}")
print("\nresults ->", OUTD)


## 3 · The harness

Written to a file rather than defined inline so it is diffable against the CPU original
(`arch_compare.py`) and so a crashed session can re-import it without re-running the
notebook from the top.


In [ ]:
%%writefile arch_gpu.py
"""Arm-3 backbone comparison, GPU port of arch_compare.py.

ONE harness. Data loading, channel construction, target definition, split, training
schedule and the free-running overlap-add rollout are identical for every run -- only the
sequence backbone changes, so differences are attributable to the architecture.

  Arm 3 only        target = u_nl - u_red,  reconstruction = u_red + SIG * Delta_hat
  70/15/15 split    model selection on val, metrics on TEST
  no physics low-pass anywhere

Deliberate differences from the CPU arch_compare.py (see the notebook's closing cell):
  1. device is cuda; tensors live on the GPU for the whole run
  2. the DataLoader is replaced by an index-batch loop over GPU-resident tensors
     (same batch size, same drop_last, so the number of optimiser steps and therefore
     the OneCycleLR schedule are unchanged -- but the shuffle RNG stream differs)
  3. the --epp physics-prior branch is removed (it needs epp_prior.py / epp_ext.py and
     is not used by this study)
  4. the 3-DOF example is removed
  5. per-run reseeding, because all runs now share one process
"""
import json
import math
import os
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
L_IN = L_OUT = 100
L_ALL = L_IN + L_OUT
HID = 96
BATCH = 64
LR = 2e-3
WD = 4e-4

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DB: dict = {}          # example -> path to .npz, set by the notebook


# ─────────────────────────── data ───────────────────────────
def load_example(name):
    """-> u_nl, u_red, u_lin, q  each (N, T, S/Sq);  P (N, k);  fs"""
    if name == "sdof":
        d = np.load(DB[name])
        g = np.where(d["converged"])[0]
        u = d["u"][g][:, :, None]
        ul = d["u_lin"][g][:, :, None]
        q = d["F"][g][:, :, None]
        P = d["params"][g]
        fs = float(d["fs"])
        u_lin = ul                              # SDOF has no reduced model: u_red == u_lin
    else:
        d = np.load(DB[name], allow_pickle=True)
        tr = lambda a: np.transpose(a, (0, 2, 1)).astype(np.float32)
        u, ul, q = tr(d["u_nl"]), tr(d["u_red"]), tr(d["q_sensor"])
        u_lin = tr(d["u_lin"])
        P = d["params"]
        fs = 1.0 / float(d["dt"]) if "dt" in d.files else 1.0 / float(d["t"][1] - d["t"][0])
    return (u.astype(np.float32), ul.astype(np.float32), u_lin.astype(np.float32),
            q.astype(np.float32), P.astype(np.float64), fs)


# ─────────────────────── backbones ───────────────────────
class GRUNet(nn.Module):
    def __init__(self, c, h=HID):
        super().__init__()
        self.net = nn.GRU(c, h, num_layers=2, batch_first=True, bidirectional=True, dropout=0.1)
        self.out_dim = h * 2
    def forward(self, x): return self.net(x)[0][:, -L_OUT:, :]


class LSTMNet(nn.Module):
    def __init__(self, c, h=HID):
        super().__init__()
        self.net = nn.LSTM(c, h, num_layers=2, batch_first=True, bidirectional=True, dropout=0.1)
        self.out_dim = h * 2
    def forward(self, x): return self.net(x)[0][:, -L_OUT:, :]


class MLPNet(nn.Module):
    """Flatten the whole window -> MLP -> reshape. No temporal inductive bias."""
    def __init__(self, c, h=HID):
        super().__init__()
        self.d = 64
        self.net = nn.Sequential(nn.Linear(L_ALL * c, 512), nn.GELU(), nn.Dropout(0.1),
                                 nn.Linear(512, 512), nn.GELU(),
                                 nn.Linear(512, L_OUT * self.d))
        self.out_dim = self.d
    def forward(self, x):
        return self.net(x.flatten(1)).view(x.shape[0], L_OUT, self.d)


class TCNNet(nn.Module):
    """Dilated causal-style conv stack with residual connections."""
    def __init__(self, c, h=HID):
        super().__init__()
        layers, cin = [], c
        for dil in (1, 2, 4, 8, 16):
            layers.append(nn.Sequential(
                nn.Conv1d(cin, h, 3, padding=dil, dilation=dil), nn.GELU(),
                nn.Conv1d(h, h, 3, padding=dil, dilation=dil), nn.GELU()))
            cin = h
        self.blocks = nn.ModuleList(layers)
        self.out_dim = h
    def forward(self, x):
        z = x.transpose(1, 2)
        for i, b in enumerate(self.blocks):
            y = b(z)[:, :, :z.shape[2]]
            z = y if i == 0 else z + y
        return z.transpose(1, 2)[:, -L_OUT:, :]


class CNN1DNet(nn.Module):
    """Plain (non-dilated) 1-D conv stack."""
    def __init__(self, c, h=HID):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(c, h, 7, padding=3), nn.GELU(),
            nn.Conv1d(h, h, 7, padding=3), nn.GELU(),
            nn.Conv1d(h, h, 7, padding=3), nn.GELU(),
            nn.Conv1d(h, h, 7, padding=3), nn.GELU())
        self.out_dim = h
    def forward(self, x): return self.net(x.transpose(1, 2)).transpose(1, 2)[:, -L_OUT:, :]


class ResBlock(nn.Module):
    def __init__(self, h, k=5):
        super().__init__()
        self.c1 = nn.Conv1d(h, h, k, padding=k // 2); self.b1 = nn.BatchNorm1d(h)
        self.c2 = nn.Conv1d(h, h, k, padding=k // 2); self.b2 = nn.BatchNorm1d(h)
    def forward(self, z):
        y = F.gelu(self.b1(self.c1(z)))
        y = self.b2(self.c2(y))
        return F.gelu(z + y)


class ResNetNet(nn.Module):
    def __init__(self, c, h=HID):
        super().__init__()
        self.stem = nn.Conv1d(c, h, 7, padding=3)
        self.blocks = nn.Sequential(*[ResBlock(h) for _ in range(4)])
        self.out_dim = h
    def forward(self, x):
        return self.blocks(F.gelu(self.stem(x.transpose(1, 2)))).transpose(1, 2)[:, -L_OUT:, :]


class TransformerNet(nn.Module):
    def __init__(self, c, h=HID):
        super().__init__()
        d = 128
        self.inp = nn.Linear(c, d)
        pe = torch.zeros(L_ALL, d)
        pos = torch.arange(L_ALL).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
        pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
        layer = nn.TransformerEncoderLayer(d, 4, dim_feedforward=256, dropout=0.1,
                                           batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, num_layers=3)
        self.out_dim = d
    def forward(self, x):
        return self.enc(self.inp(x) + self.pe)[:, -L_OUT:, :]


class MambaLikeNet(nn.Module):
    """Mamba-style selective diagonal SSM, pure PyTorch.

    mamba_ssm's fused kernel is not used here, so this reimplements the core S6
    recurrence  h_t = exp(dt_t * A) h_{t-1} + dt_t * B_t * x_t ,  y_t = C_t * h_t
    with input-dependent dt, B, C and a diagonal A, evaluated by a log-space cumulative
    scan. Label results as 'Mamba-style', not the reference kernel.
    """
    def __init__(self, c, h=HID):
        super().__init__()
        self.h = h
        self.inp = nn.Linear(c, h)
        self.dt = nn.Linear(c, h)
        self.Bp = nn.Linear(c, h)
        self.Cp = nn.Linear(c, h)
        self.A_log = nn.Parameter(torch.log(torch.rand(h) * 0.5 + 0.5))
        self.conv = nn.Conv1d(h, h, 3, padding=1, groups=h)
        self.norm = nn.LayerNorm(h)
        self.out = nn.Linear(h, h)
        self.out_dim = h
    def forward(self, x):
        u = self.inp(x)
        u = self.conv(u.transpose(1, 2)).transpose(1, 2)
        u = F.silu(u)
        dt = F.softplus(self.dt(x))                        # (B,L,H) > 0
        A = -torch.exp(self.A_log)[None, None, :]          # (1,1,H) < 0
        logdec = dt * A                                    # log of decay, <= 0
        cum = torch.cumsum(logdec, dim=1)                  # (B,L,H)
        src = dt * self.Bp(x) * u
        # h_t = exp(cum_t) * sum_{s<=t} exp(-cum_s) * src_s   (clamped for stability)
        inner = torch.cumsum(src * torch.exp(-(cum - cum.max(dim=1, keepdim=True).values)
                                             .clamp(min=-30.0)), dim=1)
        h = torch.exp((cum - cum.max(dim=1, keepdim=True).values).clamp(min=-30.0)) * inner
        y = self.Cp(x) * h
        return self.out(self.norm(y))[:, -L_OUT:, :]


class EncDecNet(nn.Module):
    """GRU encoder -> repeated context -> GRU decoder over the output window."""
    def __init__(self, c, h=HID):
        super().__init__()
        self.enc = nn.GRU(c, h, num_layers=1, batch_first=True, bidirectional=True)
        self.dec = nn.GRU(h * 2 + c, h, num_layers=1, batch_first=True)
        self.out_dim = h
    def forward(self, x):
        _, hn = self.enc(x[:, :L_IN, :])
        ctx = torch.cat([hn[0], hn[1]], dim=-1).unsqueeze(1).expand(-1, L_OUT, -1)
        return self.dec(torch.cat([ctx, x[:, -L_OUT:, :]], dim=-1))[0]


ARCHS = {"gru": GRUNet, "lstm": LSTMNet, "mlp": MLPNet, "tcn": TCNNet,
         "cnn1d": CNN1DNet, "resnet": ResNetNet, "transformer": TransformerNet,
         "mamba": MambaLikeNet, "encdec": EncDecNet}

NICE = {"gru": "GRU", "transformer": "Transformer", "mamba": "Mamba",
        "encdec": "Encoder-decoder", "lstm": "LSTM", "mlp": "MLP", "tcn": "TCN",
        "resnet": "ResNet", "cnn1d": "1D CNN"}


# ─────────────────────── head + conditioning ───────────────────────
class FiLM(nn.Module):
    def __init__(self, n_static, n_time):
        super().__init__()
        h = max(n_static * 2, 64)
        self.f = nn.Sequential(nn.Linear(n_static, h), nn.ReLU(), nn.Linear(h, n_time * 2))
    def forward(self, x, s):
        g, b = self.f(s).unsqueeze(1).chunk(2, dim=-1)
        return (1.0 + g) * x + b


class NoFiLM(nn.Module):
    """Ablation: broadcast-concat the statics instead of modulating with them."""
    def __init__(self, n_static, n_time):
        super().__init__()
        self.f = nn.Sequential(nn.Linear(n_static, 64), nn.ReLU(), nn.Linear(64, n_time))
    def forward(self, x, s): return x + self.f(s).unsqueeze(1)


class Head(nn.Module):
    def __init__(self, d, s, zero_init=True):
        super().__init__()
        self.a = nn.Sequential(nn.Linear(d, 64), nn.GELU())
        self.b = nn.Linear(64, s, bias=False)
        if zero_init:
            nn.init.zeros_(self.b.weight)      # start at Delta = 0  => Arm3 == Arm1
    def forward(self, z): return self.b(self.a(z))


class LoadSkip(nn.Module):
    """Per-sensor learned linear skip from the local load q to the local residual, with a
    gain that is a function of the static features. Zero-init so the recurrent path
    dominates early."""
    def __init__(self, n_static, sq, s):
        super().__init__()
        self.gain = nn.Sequential(nn.Linear(n_static, 64), nn.Tanh(), nn.Linear(64, s))
        nn.init.zeros_(self.gain[-1].weight); nn.init.zeros_(self.gain[-1].bias)
        self.proj = nn.Identity() if sq == s else nn.Linear(sq, s, bias=False)
    def forward(self, q_out, s_flat):
        return self.gain(s_flat).unsqueeze(1) * self.proj(q_out)


def tukey(n, a=0.3):
    w = np.ones(n, np.float32); k = int(a * n / 2)
    if k > 0:
        t = 0.5 * (1 - np.cos(np.pi * np.arange(k) / k)); w[:k] = t; w[-k:] = t[::-1]
    return w


# ─── window construction: identical for all 9 backbones, so build once per example ───
_PREP: dict = {}


def prep(example, train_frac=1.0, ar_len=None):
    """Everything upstream of the model. Cached on (example, train_frac, ar_len)."""
    key = (example, train_frac, ar_len)
    if key in _PREP:
        return _PREP[key]
    t0 = time.perf_counter()
    u_nl, u_red, u_lin, q, P, fs = load_example(example)
    N, T, S = u_nl.shape
    Sq = q.shape[2]
    step = max(L_OUT, T // 10)                       # ~10 windows per sample

    idx = np.random.default_rng(SEED).permutation(N)
    n_tr, n_va = int(0.70 * N), int(0.15 * N)
    itr_full, iva, ite = idx[:n_tr], idx[n_tr:n_tr + n_va], idx[n_tr + n_va:]
    itr = itr_full[:max(1, int(round(len(itr_full) * train_frac)))]

    qs = q.std(1, keepdims=True) + 1e-12
    qn = (q / qs).astype(np.float32)
    rs = u_red.std(1, keepdims=True) + 1e-12
    rn = (u_red / rs).astype(np.float32)
    dl = ((u_lin - u_red) / rs).astype(np.float32)
    use_dlin = float(np.abs(dl).max()) > 1e-8
    resid = (u_nl - u_red).astype(np.float32)
    SIG = resid[itr].std(axis=(0, 1), keepdims=True) + 1e-12
    tgt = (resid / SIG).astype(np.float32)

    lp = np.log(np.clip(np.abs(P) + 1e-12, 1e-30, None))
    st = ((lp - lp[itr].mean(0)) / (lp[itr].std(0) + 1e-8)).astype(np.float32)
    NS = st.shape[1]

    tl = np.arange(L_ALL, dtype=np.float32) / fs
    ps = np.sin(2 * np.pi * 1.0 * tl)[None, :, None].astype(np.float32)
    pc = np.cos(2 * np.pi * 1.0 * tl)[None, :, None].astype(np.float32)

    AR_LEN = L_IN if ar_len is None else max(0, min(int(ar_len), L_IN))
    starts = [s_ for s_ in range(0, T, step) if s_ + L_ALL <= T + L_IN]
    z = np.zeros((L_IN, S), np.float32); zq = np.zeros((L_IN, Sq), np.float32)

    def build(ids):
        X, SS, Y = [], [], []
        for i in ids:
            qp = np.concatenate([zq, qn[i]]); rp = np.concatenate([z, rn[i]])
            dp = np.concatenate([z, dl[i]]);  tp = np.concatenate([z, tgt[i]])
            for s_ in starts:
                e = s_ + L_ALL
                up = np.concatenate([tp[s_:s_ + L_IN], np.zeros((L_OUT, S), np.float32)])
                if AR_LEN < L_IN:              # keep only the last AR_LEN steps of context
                    up = up.copy(); up[:L_IN - AR_LEN] = 0.0
                ch = [qp[s_:e], rp[s_:e]] + ([dp[s_:e]] if use_dlin else []) + \
                     [up, ps[0], pc[0]]
                X.append(np.concatenate(ch, 1)); SS.append(st[i]); Y.append(tp[s_ + L_IN:e])
        return (np.asarray(X, np.float32), np.asarray(SS, np.float32), np.asarray(Y, np.float32))

    Xtr, Str, Ytr = build(itr)
    Xva, Sva, Yva = build(iva)
    g = lambda a: torch.from_numpy(a).to(DEV)
    out = dict(Xtr=g(Xtr), Str=g(Str), Ytr=g(Ytr), Xva=g(Xva), Sva=g(Sva), Yva=g(Yva),
               C=Xtr.shape[2], NS=NS, S=S, Sq=Sq, T=T, N=N, fs=fs,
               itr=itr, iva=iva, ite=ite, use_dlin=use_dlin, AR_LEN=AR_LEN,
               qn=qn, rn=rn, dl=dl, st=st, SIG=SIG, ps=ps, pc=pc,
               u_nl=u_nl, u_red=u_red, n_win=len(starts),
               prep_sec=round(time.perf_counter() - t0, 1))
    _PREP.clear()                  # one example at a time: the GPU tensors are large
    _PREP[key] = out
    print(f"  prep {example}: N={N} T={T} S={S} channels={out['C']} statics={NS} "
          f"| train {len(itr)} val {len(iva)} test {len(ite)} "
          f"| {out['Xtr'].shape[0]} train windows ({out['n_win']}/sample) "
          f"| {out['prep_sec']}s")
    return out


def run_one(example, arch, epochs=10, train_frac=1.0, no_film=False, no_skip=False,
            no_zeroinit=False, ar_len=None, save_ckpt=None):
    """Train one backbone on one example and score it on the TEST split. -> result dict"""
    torch.manual_seed(SEED); np.random.seed(SEED)
    if DEV.type == "cuda":
        torch.cuda.manual_seed_all(SEED)
    D = prep(example, train_frac, ar_len)
    C, NS, S, Sq, T = D["C"], D["NS"], D["S"], D["Sq"], D["T"]

    film = (NoFiLM if no_film else FiLM)(NS, C).to(DEV)
    back = ARCHS[arch](C).to(DEV)
    head = Head(back.out_dim, S, zero_init=not no_zeroinit).to(DEV)
    mods = [film, back, head]
    skip = None
    if not no_skip:
        skip = LoadSkip(NS, Sq, S).to(DEV); mods.append(skip)
    mods = tuple(mods)

    def fwd(xb, sb):
        y = head(back(film(xb, sb)))
        return y if skip is None else y + skip(xb[:, -L_OUT:, :Sq], sb)
    par = [p for m in mods for p in m.parameters()]
    npar = sum(p.numel() for p in par)

    Xtr, Str, Ytr = D["Xtr"], D["Str"], D["Ytr"]
    Xva, Sva, Yva = D["Xva"], D["Sva"], D["Yva"]
    ntr = Xtr.shape[0]
    steps = max(1, ntr // BATCH)                        # == len(DataLoader(drop_last=True))
    opt = torch.optim.AdamW(par, lr=LR, weight_decay=WD)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, epochs=epochs,
                                              steps_per_epoch=steps, pct_start=0.3)
    crit = nn.MSELoss()
    best, bstate, t0 = float("inf"), None, time.perf_counter()
    for ep in range(epochs):
        for m in mods: m.train()
        order = torch.randperm(ntr, device=DEV)
        for k in range(steps):
            j = order[k * BATCH:(k + 1) * BATCH]
            opt.zero_grad(set_to_none=True)
            loss = crit(fwd(Xtr[j], Str[j]), Ytr[j])
            loss.backward(); nn.utils.clip_grad_norm_(par, 1.0); opt.step(); sch.step()
        for m in mods: m.eval()
        tot, nva = 0.0, Xva.shape[0]
        with torch.no_grad():
            for k in range(0, nva, BATCH):
                sl = slice(k, min(k + BATCH, nva))
                tot += crit(fwd(Xva[sl], Sva[sl]), Yva[sl]).item() * (sl.stop - sl.start)
        v = tot / nva
        if v < best:
            best = v
            bstate = [{k: t.detach().clone() for k, t in m.state_dict().items()} for m in mods]
        print(f"    ep {ep+1:2d}/{epochs}  train {loss.item():.5f}  val {v:.5f}"
              f"{'  *' if v == best else ''}", flush=True)
    for m, s_ in zip(mods, bstate): m.load_state_dict(s_)
    wall = time.perf_counter() - t0

    # ── free-running overlap-add rollout on TEST ──
    for m in mods: m.eval()
    ite, hop, tap = D["ite"], max(1, L_OUT // 8), tukey(L_OUT)
    qn, rn, dl, st = D["qn"], D["rn"], D["dl"], D["st"]
    ps, pc, use_dlin, AR_LEN = D["ps"], D["pc"], D["use_dlin"], D["AR_LEN"]
    out = np.zeros((len(ite), T, S), np.float32)
    with torch.no_grad():
        for b0 in range(0, len(ite), 16):
            ci = ite[b0:b0 + 16]; nb = len(ci)
            qp = np.concatenate([np.zeros((nb, L_IN, Sq), np.float32), qn[ci]], 1)
            rp = np.concatenate([np.zeros((nb, L_IN, S), np.float32), rn[ci]], 1)
            dp = np.concatenate([np.zeros((nb, L_IN, S), np.float32), dl[ci]], 1)
            acc = np.zeros((nb, T + L_IN, S), np.float32)
            wg = np.zeros((nb, T + L_IN, 1), np.float32)
            est = np.zeros((nb, T + L_IN, S), np.float32)
            sb = torch.from_numpy(st[ci]).to(DEV)
            for s_ in range(0, T, hop):
                e = s_ + L_ALL
                if e > T + L_IN: break
                up = np.concatenate([est[:, s_:s_ + L_IN],
                                     np.zeros((nb, L_OUT, S), np.float32)], 1)
                if AR_LEN < L_IN: up[:, :L_IN - AR_LEN] = 0.0
                ch = [qp[:, s_:e], rp[:, s_:e]] + ([dp[:, s_:e]] if use_dlin else []) + \
                     [up, np.repeat(ps, nb, 0), np.repeat(pc, nb, 0)]
                xb = torch.from_numpy(np.concatenate(ch, 2)).to(DEV)
                pr = fwd(xb, sb).cpu().numpy()
                acc[:, s_ + L_IN:e] += pr * tap[None, :, None]
                wg[:, s_ + L_IN:e] += tap[None, :, None]
                sw = np.where(wg[:, s_ + L_IN:e] > 0, wg[:, s_ + L_IN:e], 1.0)
                est[:, s_ + L_IN:e] = acc[:, s_ + L_IN:e] / sw
            sw = np.where(wg > 0, wg, 1.0)
            out[b0:b0 + nb] = (acc / sw)[:, L_IN:L_IN + T]

    u_nl, u_red = D["u_nl"], D["u_red"]
    pred = u_red[ite] + out * D["SIG"]
    tr_ = u_nl[ite]
    r2 = float(1 - ((tr_ - pred) ** 2).sum() / (((tr_ - tr_.mean()) ** 2).sum() + 1e-12))
    r2b = float(1 - ((tr_ - u_red[ite]) ** 2).sum() / (((tr_ - tr_.mean()) ** 2).sum() + 1e-12))
    # unweighted mean over sensors, for the tunnel and the turbine where the sensors span
    # decades of energy and the pooled value hides the quiet ones
    per = [float(1 - ((tr_[:, :, s] - pred[:, :, s]) ** 2).sum()
                 / (((tr_[:, :, s] - tr_[:, :, s].mean()) ** 2).sum() + 1e-12))
           for s in range(S)]
    per1 = [float(1 - ((tr_[:, :, s] - u_red[ite][:, :, s]) ** 2).sum()
                  / (((tr_[:, :, s] - tr_[:, :, s].mean()) ** 2).sum() + 1e-12))
            for s in range(S)]
    pk_t, pk_p = np.max(np.abs(tr_), 1), np.max(np.abs(pred), 1)
    if save_ckpt:
        # atomic: a session killed mid-write must not leave a truncated .pt beside a
        # valid .json, which would read as a complete run
        torch.save({"film": film.state_dict(), "back": back.state_dict(),
                    "head": head.state_dict(),
                    "skip": (skip.state_dict() if skip is not None else None),
                    "arch": arch, "example": example, "SEED": SEED, "C": C, "S": S,
                    "no_film": no_film, "no_skip": no_skip, "no_zeroinit": no_zeroinit,
                    "SIG": D["SIG"], "itr": D["itr"], "iva": D["iva"], "ite": D["ite"]},
                   save_ckpt + ".tmp")
        os.replace(save_ckpt + ".tmp", save_ckpt)
    return dict(example=example, arch=arch, R2=r2, R2_arm1=r2b, gain=r2 - r2b,
                R2_unweighted=float(np.mean(per)), R2_arm1_unweighted=float(np.mean(per1)),
                r2_per_sensor=per, val_mse=best, params=npar, sec=round(wall, 1),
                peak_err=float((np.abs(pk_t - pk_p) / (pk_t + 1e-30) * 100).mean()),
                n_train=len(D["itr"]), n_val=len(D["iva"]), n_test=len(ite),
                channels=C, epochs=epochs, train_frac=train_frac, seed=SEED,
                device=DEV.type, gpu=(torch.cuda.get_device_name(0)
                                     if DEV.type == "cuda" else None))


In [ ]:
import importlib
import arch_gpu as A
importlib.reload(A)
print('harness on', A.DEV, '| backbones:', ', '.join(A.ARCHS))


## 4 · Configuration

Copying the data onto the local disk first matters: Drive is a network mount, and the
tunnel database alone is 172 MB read once per example.


In [ ]:
import os, shutil, time

LOCAL = "/content/db"
os.makedirs(LOCAL, exist_ok=True)
t0 = time.perf_counter()
for k, p in DB_PATHS.items():
    dst = os.path.join(LOCAL, os.path.basename(p))
    if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(p):
        shutil.copy2(p, dst)
    A.DB[k] = dst
print(f"databases staged locally in {time.perf_counter()-t0:.0f}s")

EXAMPLES = ["sdof", "frame3", "tunnel", "turbine"]
BACKBONES = ["gru", "lstm", "transformer", "mamba", "encdec", "tcn", "cnn1d", "resnet", "mlp"]
EPOCHS = 10
TRAIN_FRAC = 1.0          # 1.0 = all of the 70% train split;  1/3 = the earlier CPU budget
SAVE_CKPT = True          # a .pt per run (~1-20 MB each) so the table is re-scorable later

print(f"\n{len(EXAMPLES)} examples x {len(BACKBONES)} backbones = "
      f"{len(EXAMPLES)*len(BACKBONES)} runs, {EPOCHS} epochs, seed {A.SEED}, "
      f"train_frac {TRAIN_FRAC}, save_ckpt {SAVE_CKPT}")


## 5 · Run

Ordered example-outer so the windowed tensors are built once per example and reused by all
nine backbones.

**Resumable.** Every finished run writes `results/<example>_<arch>.json` (plus a `.pt` if
`SAVE_CKPT`), and re-entering this cell skips anything already on disk. So if Colab
disconnects, reclaims the GPU, or you hit the session limit: re-run cells 1–4 to remount and
re-stage, then re-run this cell. It picks up where it stopped and you lose at most the one
run that was in flight.

Writes are atomic — JSON goes to a `.tmp` and is renamed — and an existing JSON that will
not parse is treated as unfinished and re-run, so a kill in the middle of a write cannot
leave a corrupt file that is silently skipped forever.


In [ ]:
import json, os, time, traceback

def finished(ex, ar):
    """A run counts as done only if its JSON parses and carries the metric."""
    jf = os.path.join(OUTD, f"{ex}_{ar}.json")
    if not os.path.exists(jf):
        return False
    try:
        r = json.load(open(jf))
        return "R2" in r and "sec" in r
    except Exception:
        print(f"   corrupt {os.path.basename(jf)} -> will re-run")
        return False


def save_atomic(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w") as fh:
        json.dump(obj, fh, indent=1)
        fh.flush(); os.fsync(fh.fileno())
    os.replace(tmp, path)


t_start = time.perf_counter()
todo = [(e, a) for e in EXAMPLES for a in BACKBONES]
pre = [p for p in todo if finished(*p)]
print(f"{len(pre)}/{len(todo)} already done, {len(todo)-len(pre)} to go\n")
n_new, t_new = 0, 0.0

for n, (ex, ar) in enumerate(todo, 1):
    if finished(ex, ar):
        print(f"[{n:2d}/{len(todo)}] {ex:8s} {ar:12s} SKIP (done)")
        continue
    print(f"[{n:2d}/{len(todo)}] {ex:8s} {ar:12s} ...", flush=True)
    try:
        ck = os.path.join(OUTD, f"{ex}_{ar}.pt") if SAVE_CKPT else None
        r = A.run_one(ex, ar, epochs=EPOCHS, train_frac=TRAIN_FRAC, save_ckpt=ck)
    except KeyboardInterrupt:
        print("\ninterrupted -- re-run this cell to continue from here"); break
    except Exception:
        traceback.print_exc()
        print(f"   !! {ex}/{ar} FAILED -- continuing\n")
        continue
    save_atomic(r, os.path.join(OUTD, f"{ex}_{ar}.json"))
    with open(os.path.join(OUTD, "progress.csv"), "a") as fh:      # append-only audit trail
        fh.write(f"{ex},{ar},{r['R2']:.6f},{r['R2_unweighted']:.6f},{r['sec']:.1f},"
                 f"{r['params']},{time.strftime('%Y-%m-%d %H:%M:%S')}\n")
    n_new += 1; t_new = time.perf_counter() - t_start
    rem = (len(todo) - len(pre) - n_new) * t_new / n_new
    print(f"   R2={r['R2']:+.4f} (arm1 {r['R2_arm1']:+.4f}, gain {r['gain']:+.4f})  "
          f"unweighted {r['R2_unweighted']:+.4f}  params={r['params']:,}  {r['sec']:.0f}s"
          f"   | {n_new} done this session in {t_new/60:.0f}m, ETA {rem/60:.0f}m\n", flush=True)

nd = sum(finished(*p) for p in todo)
print(f"{nd}/{len(todo)} complete. {(time.perf_counter()-t_start)/60:.0f} min this session.")
if nd < len(todo):
    print("Not finished -- re-run cells 1-4 then this cell to continue.")


## 6 · The table

Wins is the per-column maximum. Parameters and Time vary by example because the channel
count does, so both are reported as the frame value and the 4-example sum respectively —
stated explicitly, since mixing scopes silently is easy to do here.


In [ ]:
import csv, glob, json, os

import numpy as np

R = {}
for f in glob.glob(os.path.join(OUTD, "*.json")):
    r = json.load(open(f)); R[(r["example"], r["arch"])] = r
have = [a for a in BACKBONES if all((e, a) in R for e in EXAMPLES)]
miss = [a for a in BACKBONES if a not in have]
if miss:
    print(f"incomplete, excluded from the table: {miss}\n")

TITLE = {"sdof": "SDOF", "frame3": "Frame", "tunnel": "Tunnel", "turbine": "Turbine"}

def table(metric, fname):
    best = {e: max(have, key=lambda a: R[(e, a)][metric]) for e in EXAMPLES}
    rows = []
    for a in have:
        v = [R[(e, a)][metric] for e in EXAMPLES]
        rows.append(dict(arch=A.NICE[a], key=a, vals=v, mean=float(np.mean(v)),
                         wins=sum(best[e] == a for e in EXAMPLES),
                         params=R[("frame3", a)]["params"],
                         sec=sum(R[(e, a)]["sec"] for e in EXAMPLES)))
    rows.sort(key=lambda r: -r["mean"])
    hdr = ["Backbone"] + [TITLE[e] for e in EXAMPLES] + ["Mean", "Wins", "Parameters", "Time (s)"]
    print(f"--- {metric}  (seed {A.SEED}, {EPOCHS} epochs, train_frac {TRAIN_FRAC}, TEST split)")
    print("".join(h.rjust(11) if i else h.ljust(17) for i, h in enumerate(hdr)))
    for r in rows:
        print(r["arch"].ljust(17) + "".join(f"{x:>+11.4f}" for x in r["vals"])
              + f"{r['mean']:>+11.4f}{r['wins']:>11d}{r['params']/1e3:>10.0f}k{r['sec']:>11.0f}")
    spread = [max(R[(e, a)][metric] for a in have) - min(R[(e, a)][metric] for a in have)
              for e in EXAMPLES]
    print("Spread".ljust(17) + "".join(f"{x:>11.3f}" for x in spread))
    print("Arm 1".ljust(17) + "".join(f"{R[(e, have[0])]['R2_arm1' if metric == 'R2' else 'R2_arm1_unweighted']:>+11.4f}" for e in EXAMPLES))
    with open(os.path.join(OUTD, fname), "w", newline="") as fh:
        w = csv.writer(fh); w.writerow(["backbone"] + list(EXAMPLES) + ["mean", "wins", "params", "sec_total"])
        for r in rows:
            w.writerow([r["key"]] + [f"{x:.4f}" for x in r["vals"]]
                       + [f"{r['mean']:.4f}", r["wins"], r["params"], round(r["sec"])])
    print(f"-> {os.path.join(OUTD, fname)}\n")

table("R2", "arch_table_pooled.csv")
table("R2_unweighted", "arch_table_unweighted.csv")
print("n_train/n_test per example:")
for e in EXAMPLES:
    r = R[(e, have[0])]
    print(f"  {e:8s} train {r['n_train']:5d}  val {r['n_val']:4d}  test {r['n_test']:4d}"
          f"  channels {r['channels']:3d}")


---

## What this table can and cannot say

**Single seed.** One seed gives no error bar. On the earlier CPU sweep the top five
backbones sat within 0.013 of each other at full data, which is almost certainly inside
seed noise — so read this as *"no backbone is clearly better"* unless a gap is large. If a
ranking has to be defended, re-run with `SEED` 0/1/2 (edit `A.SEED` and use a different
`OUTD`) and report mean ± spread.

**GPU results will not be bit-identical to the CPU sweep**, by design:

1. cuDNN's RNN backward pass is non-deterministic, so GRU / LSTM / Encoder–decoder move a
   little between repeat runs on the same seed. The convolutional and attention backbones
   are closer but still not exact.
2. The DataLoader was replaced by an index-batch loop over GPU-resident tensors. Batch size,
   `drop_last` and therefore the optimiser-step count and the OneCycleLR schedule are
   unchanged, but the shuffle order differs.

Everything that defines the *experiment* — split, seeding of the split, channels, target
normalisation, epochs, LR schedule, model selection on val, rollout hop and taper, and the
metric — is unchanged from `arch_compare.py`.

**Two metrics are reported.** Pooled R² is what the earlier table used. For the tunnel and
the turbine the sensors span decades of energy, so pooling hides the quiet ones; the
unweighted mean over sensors is the harder and more honest number there. They disagree
substantially — quote whichever the surrounding text defines, and say which.

**The databases differ from the earlier CPU sweep** for two of the four rows: SDOF is the
1500-sample η ∈ [0.60, 0.96] release database (the old sweep used a different 1000-sample
one whose Arm 1 was ≈ 0, which is why SDOF read ≈ 0 there), and the turbine is the revised
60 s V_hub ≤ 20.2 m/s example (the old sweep used the superseded 300-sample 600 s one).
Both changes make this table consistent with the paper's system table and 3-arm table, and
mean the numbers are **not** comparable run-for-run with the earlier one.
